# 04. 데이터 엔지니어링 (변수 생성·인코딩·레이블링)

| 항목 | 내용 |
|---|---|
| 목적 | EDA에서 확인한 신호를 모델이 쓸 수 있는 변수로 만들고, 중복 변수를 정리해 **피해 예측·지원금 최적화의 입력 테이블**을 만든다 |
| 입력 | `data/processed/` (01 전처리 산출물) |
| 출력 | `data/processed/model_daily_group.parquet` (모델 학습용), `data/processed/unit_profile.parquet` (지역 × 업종 구조 변수), `references/feature_dictionary.csv` (변수 사전) |
| 브랜치 | `ana-engin` |

**EDA → 엔지니어링 대응**
| EDA에서 확인한 것 | 만드는 변수 |
|---|---|
| 기온 계열·강수 계열 변수끼리 상관 0.94~1.00 | 계열별 **대표 변수** 선정, 나머지 제외 (VIF로 확인) |
| 폭염 효과는 휴가철 효과 → 달력 통제 필수 | 달력 변수(요일·공휴일·연휴 전후·월급주간), **최근 매출 수준**(지난주 같은 요일, 직전 7일 평균) |
| 비는 내리는 시간대의 매출을 깎음 | **시간대별 강수**, 업종의 시간대 매출 비중으로 가중한 **영업시간 강수 노출** |
| 비가 많을수록 피해 증가, 연속 강수 | 강수 구간, log 강수량, **전날 비**, 연속 강수일 |
| 춘천·외지 고객·업종별로 반응이 다름 | 지역·업종 **인코딩**, 업종 구조 변수(외지 비중·독립점포 비중 등), **기상 × 구조** 상호작용 |
| 기상 유형별 업종 반응 (가설 3) | 기상 유형·특보 수준 **레이블** |

**진행 순서**
1. 환경 설정·입력
2. 기상 변수 정리 — 대표 변수 선정 (VIF)
3. 기상 변수 생성 — 구간·시간대·지속성·평년 편차
4. 레이블링 — 기상 유형·특보 수준
5. 달력·매출 수준 변수 (누수 방지)
6. 지역 × 업종 구조 변수 (unit profile)
7. 인코딩·상호작용
8. 최종 테이블 점검·저장, 변수 사전

## 1. 환경 설정·입력

In [1]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:,.3f}'.format)

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROC = ROOT / 'data' / 'processed'
REF = ROOT / 'references'
load = lambda name: pd.read_parquet(PROC / f'{name}.parquet')

INK, INK2, GRID, SURFACE = '#0b0b0b', '#52514e', '#e4e3df', '#fcfcfb'
REGION_COLOR = {'강남구': '#2a78d6', '춘천시': '#eb6834'}
plt.rcParams.update({
    'font.family': 'Malgun Gothic', 'axes.unicode_minus': False, 'figure.dpi': 110,
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'axes.edgecolor': GRID,
    'axes.labelcolor': INK2, 'xtick.color': INK2, 'ytick.color': INK2,
    'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.titlecolor': INK, 'axes.titlelocation': 'left',
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.6, 'axes.axisbelow': True,
    'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False,
})
SLOTS = ['00_05', '06_11', '12_17', '18_23']

mart = load('mart_daily_group')            # 날짜 × 지역 × 업종대분류 매출 (기상 미결합)
weather = load('weather_daily')
wslot = load('weather_slot')
normal = load('weather_normal')
slot_sales = load('mart_slot_group')
origin = load('mart_origin_group')
store = load('store_group_summary')
flow_rm = load('flow_region_month')
print('mart', mart.shape, '| weather', weather.shape, '| weather_slot', wslot.shape)

mart (5520, 13) | weather (368, 29) | weather_slot (1472, 15)


## 2. 기상 변수 정리 — 대표 변수 선정

EDA에서 기온 계열(평균·최고·최저·체감)과 강수 계열(일합계·1시간·3시간 최대·강수 시간)은 서로 상관이 0.94~1.00이었다.
**분산팽창계수(VIF)**로 다중공선성을 확인하고 계열마다 대표 변수 하나를 남긴다. VIF가 10을 넘으면 다른 변수들로 거의 설명된다는 뜻이다.

In [2]:
def vif_table(df, cols):
    X = df[cols].dropna().astype(float)
    X = (X - X.mean()) / X.std()
    X.insert(0, 'const', 1.0)
    return pd.Series([variance_inflation_factor(X.values, i + 1) for i in range(len(cols))], index=cols, name='VIF')

all_vars = ['temp_mean', 'temp_max', 'temp_min', 'feels_like_max', 'humidity_mean', 'wind_mean',
            'rain_sum', 'rain_max_1h', 'rain_max_3h', 'rain_hours', 'snow_new']
chosen = ['feels_like_max', 'temp_min', 'wind_mean', 'rain_sum', 'snow_new']
pd.concat({'전체 후보': vif_table(weather, all_vars), '선정 변수': vif_table(weather, chosen)}, axis=1).round(1)

,전체 후보,선정 변수
temp_mean,465.600,NaN
temp_max,251.100,NaN
temp_min,185.900,16.700
feels_like_max,190.500,16.100
humidity_mean,2.900,NaN
wind_mean,1.900,1.200
rain_sum,16.600,1.100
rain_max_1h,12.800,NaN
rain_max_3h,25.200,NaN
rain_hours,3.600,NaN


| 계열 | 대표 변수 | 이유 | 제외 |
|---|---|---|---|
| 더위 | `feels_like_max` (일최고 체감온도) | 폭염 판정 기준(기상청)이고 습도를 이미 반영 | temp_mean, temp_max, humidity_mean |
| 추위 | `temp_min` (일최저기온) | 한파 판정 기준. 여름에는 최고 체감온도와 상관이 높지만 계절 변수를 구간·편차로 바꾸면 겹침이 줄어든다 | — |
| 바람 | `wind_mean` | 다른 변수와 상관 낮음 | — |
| 비 | `rain_sum` (일강수량) → 구간·log·시간대로 변환 | 강도(1·3시간 최대)·지속(강수 시간)과 상관 0.98 이상 | rain_max_1h, rain_max_3h, rain_hours |
| 눈 | `snow_new` (신적설) | 다른 변수와 독립 | — |

`feels_like_max`와 `temp_min`은 둘 다 기온이라 원값끼리는 상관이 높다. 그래서 모델에는 원값보다 **구간(폭염·한파)과 평년 대비 편차**로 넣는다 (3절).

## 3. 기상 변수 생성
### 3-1. 구간·변환·평년 편차

In [3]:
w = weather[['date', 'region', 'feels_like_max', 'temp_min', 'wind_mean', 'rain_sum', 'snow_new', 'snow_depth_max',
             'rain_max_3h']].copy()
w['month_day'] = w['date'].dt.strftime('%m-%d')
w = w.merge(normal[['region', 'month_day', 'normal_feels_like_max', 'normal_temp_min', 'normal_rain_sum']],
            on=['region', 'month_day'], how='left').drop(columns='month_day')

# 더위·추위 구간 (기상청 특보 기준)
w['heat_30_33'] = w['feels_like_max'].between(30, 33, inclusive='left').astype(int)
w['heat_33'] = (w['feels_like_max'] >= 33).astype(int)
w['heat_excess'] = (w['feels_like_max'] - 30).clip(lower=0)            # 30℃ 초과분 (연속형)
w['cold'] = (w['temp_min'] <= -12).astype(int)
w['cold_excess'] = (-5 - w['temp_min']).clip(lower=0)                  # −5℃ 미만 추위 정도 (연속형)
# 비 구간 + 연속형
w['rain_light'] = ((w['rain_sum'] > 0) & (w['rain_sum'] < 5)).astype(int)
w['rain_mid'] = ((w['rain_sum'] >= 5) & (w['rain_sum'] < 30)).astype(int)
w['rain_heavy'] = (w['rain_sum'] >= 30).astype(int)
w['log_rain'] = np.log1p(w['rain_sum'])
# 눈
w['snow'] = (w['snow_new'] >= 0.5).astype(int)
w['snow_cover'] = (w['snow_depth_max'] > 0).astype(int)                # 쌓인 눈이 남아 있는 날
# 평년 대비 편차
w['anom_feels_like'] = w['feels_like_max'] - w['normal_feels_like_max']
w['anom_temp_min'] = w['temp_min'] - w['normal_temp_min']
w['anom_rain'] = w['rain_sum'] - w['normal_rain_sum']
w[['heat_30_33', 'heat_33', 'cold', 'rain_light', 'rain_mid', 'rain_heavy', 'snow', 'snow_cover']].groupby(w['region']).sum()

,heat_30_33,heat_33,cold,rain_light,rain_mid,rain_heavy,snow,snow_cover
region,,,,,,,,
강남구,35,25,0,29,25,13,2,6
춘천시,34,19,3,25,19,17,3,11


In [4]:
derived = ['heat_30_33', 'heat_33', 'cold', 'anom_feels_like', 'anom_temp_min', 'wind_mean', 'log_rain', 'snow']
print('모델에 넣을 기상 파생 변수의 VIF (모두 10 미만이면 다중공선성 문제 없음)')
vif_table(w, derived).round(2).to_frame().T

모델에 넣을 기상 파생 변수의 VIF (모두 10 미만이면 다중공선성 문제 없음)


,heat_30_33,heat_33,cold,anom_feels_like,anom_temp_min,wind_mean,log_rain,snow
VIF,1.110,1.210,1.130,2.130,1.800,1.110,1.170,1.090


- 원값 11개를 모두 쓰면 VIF가 최대 466으로, 기온·강수 변수가 서로를 거의 완전히 설명한다.
- 대표 변수 5개로 줄여도 기온 두 개(체감 최고·최저기온)는 VIF 16으로 여전히 높다. 하지만 이를 **구간(폭염·한파)과 평년 편차**로 바꾸면 모든 변수의 VIF가 10 아래로 내려간다 → 모델에는 파생 변수를 쓴다.

### 3-2. 시간대별 강수와 '영업시간 강수 노출'
EDA 5-2절: 비는 **내리는 시간대**의 매출을 줄였다. 업종마다 매출이 몰리는 시간이 다르므로(주점 = 자정 이후, 의료 = 오전),
같은 비라도 **그 업종이 장사하는 시간에 온 비**가 더 중요하다.

$$\text{rain\_exposure}_{r,g,t} = \sum_{s \in \text{시간대}} \text{매출비중}_{r,g,s} \times \text{강수량}_{r,t,s}$$

- 매출비중: 지역 × 업종의 6개월 시간대별 매출 비중 (고정값 — 날짜별 비중을 쓰면 그날 매출 정보가 새어 들어간다)

In [5]:
rs = wslot.pivot_table(index=['date', 'region'], columns='time_slot', values='slot_rain_sum', observed=True)
rs.columns = [f'rain_{c}' for c in rs.columns]
rs = rs.reset_index()

share = slot_sales.groupby(['region', 'industry_group', 'time_slot'], observed=True)['amount'].sum()
share = (share / share.groupby(level=[0, 1]).transform('sum')).unstack('time_slot')
share.columns = [f'sales_share_{c}' for c in share.columns]
share = share.reset_index()
print('업종 시간대 매출 비중 예시 (지역 × 업종 30개 중 일부)')
display(share.set_index(['region', 'industry_group']).loc[[('춘천시', '주점·유흥'), ('춘천시', '의료·보건'), ('강남구', '음식점')]].round(2))

업종 시간대 매출 비중 예시 (지역 × 업종 30개 중 일부)


sales_share_00_05  sales_share_06_11  sales_share_12_17  sales_share_18_23
region industry_group                                                                            
춘천시    주점·유흥                       0.520              0.010              0.010              0.460
       의료·보건                       0.010              0.430              0.510              0.060
강남구    음식점                         0.060              0.090              0.380              0.470

### 3-3. 지속성 — 전날 비, 연속 강수일, 연속 폭염일
- `rain_prev`: 전날 강수량 (비 온 다음 날 반등·지연 효과)
- `rain_streak`: 오늘까지 연속으로 비(1mm↑)가 온 날 수
- `heat_streak`: 오늘까지 연속 폭염(체감 33℃↑)일 수 — 폭염특보는 2일 지속 조건이 있음

In [6]:
w = w.merge(rs, on=['date', 'region'], how='left').sort_values(['region', 'date']).reset_index(drop=True)
g = w.groupby('region')
w['rain_prev'] = g['rain_sum'].shift(1).fillna(0)
w['log_rain_prev'] = np.log1p(w['rain_prev'])

def streak(flag):
    out, run = [], 0
    for f in flag:
        run = run + 1 if f else 0
        out.append(run)
    return out

w['rain_streak'] = g['rain_sum'].transform(lambda s: streak(s >= 1))
w['heat_streak'] = g['heat_33'].transform(lambda s: streak(s == 1))
print('연속 강수일 최대:', w.groupby('region')['rain_streak'].max().to_dict(),
      '| 연속 폭염일 최대:', w.groupby('region')['heat_streak'].max().to_dict())

연속 강수일 최대: {'강남구': 7, '춘천시': 7} | 연속 폭염일 최대: {'강남구': 11, '춘천시': 9}


## 4. 레이블링 — 기상 유형·특보 수준

| 레이블 | 값 | 기준 |
|---|---|---|
| `weather_type` | 눈 / 강한 비 / 폭염 / 한파 / 비 / 평상 | 하루 하나의 유형 (겹치면 드문 사건 우선) — 전처리와 동일 |
| `heat_alert` | 0 없음 / 1 주의보급 / 2 경보급 | 일최고 체감 33℃ / 35℃ |
| `rain_alert` | 0 / 1 / 2 | 3시간 강수 60mm / 90mm (호우특보 3시간 기준) 또는 일강수 80mm / 150mm |
| `cold_alert` | 0 / 1 / 2 | 일최저 −12℃ / −15℃ |
| `snow_alert` | 0 / 1 / 2 | 신적설 5cm / 20cm (대설특보) |
| `alert_level` | 0 / 1 / 2 | 위 네 가지 중 최댓값 — 지원 발동 단계 설계용 |

특보는 실제 발령 기록이 아니라 **관측값이 특보 기준을 넘었는지**로 근사한 것이다 (지속 시간·전날 대비 조건 미반영).

In [7]:
w = w.merge(weather[['date', 'region', 'weather_type']], on=['date', 'region'], how='left')
w['heat_alert'] = np.select([w['feels_like_max'] >= 35, w['feels_like_max'] >= 33], [2, 1], 0)
w['rain_alert'] = np.select([(w['rain_max_3h'] >= 90) | (w['rain_sum'] >= 150), (w['rain_max_3h'] >= 60) | (w['rain_sum'] >= 80)], [2, 1], 0)
w['cold_alert'] = np.select([w['temp_min'] <= -15, w['temp_min'] <= -12], [2, 1], 0)
w['snow_alert'] = np.select([w['snow_new'] >= 20, w['snow_new'] >= 5], [2, 1], 0)
w['alert_level'] = w[['heat_alert', 'rain_alert', 'cold_alert', 'snow_alert']].max(axis=1)
alerts = w.melt(id_vars='region', value_vars=['heat_alert', 'rain_alert', 'cold_alert', 'snow_alert', 'alert_level'])
alerts.groupby(['region', 'variable', 'value']).size().unstack([0, 2], fill_value=0)

region       강남구         춘천시       
value          0   1  2    0   1  2
variable                           
alert_level  157  19  8  160  21  3
cold_alert   184   0  0  181   3  0
heat_alert   159  17  8  165  16  3
rain_alert   183   1  0  183   1  0
snow_alert   183   1  0  183   1  0

- 폭염 특보급 날은 강남 25일(경보급 8일)·춘천 19일(경보급 3일)로 충분하다.
- 호우·대설 특보급은 지역별 1일, 한파는 춘천 3일뿐이다. 이 레이블들은 모델 입력보다는 **지원 발동 단계(시나리오) 설계**와 결과 해석에 쓴다.

## 5. 달력·매출 수준 변수 (누수 방지)

피해 예측은 "**내일 비가 오면 매출이 얼마나 줄까**"를 전날 판단하는 데 쓰인다. 그래서 매출 수준 변수는 **예측 시점(전날)에 이미 알 수 있는 값만** 쓴다.

| 변수 | 정의 | 누수 여부 |
|---|---|---|
| `lag7` | 7일 전(지난주 같은 요일) log 매출 | 과거 값 — 안전 |
| `lag14` | 14일 전 log 매출 | 안전 |
| `roll7` | 직전 7일(t−1 ~ t−7) log 매출 평균 | 당일 제외 — 안전 |
| `roll7_offday_gap` | 직전 7일 중 휴일 평균 − 평일 평균 (업종의 휴일 민감도) | 안전 |
| `dow`, `is_weekend`, `is_holiday`, `is_offday` | 요일·주말·공휴일·휴일 | 달력 — 안전 |
| `holiday_prev`, `holiday_next` | 전날·다음날이 공휴일 (연휴 전후) | 달력 — 안전 |
| `is_payday_week` | 25일~말일 | 달력 — 안전 |
| `week`, `month` | 주·월 번호 (회귀의 고정효과용) | 달력 — 안전 |

첫 14일(2025-07-01 ~ 07-14)은 `lag14`가 없으므로 **학습에서 제외 표시**(`use_for_model = False`)만 하고 행은 남긴다.

In [8]:
df = mart.merge(w, on=['date', 'region'], how='left', validate='many_to_one')
df = df.sort_values(['region', 'industry_group', 'date']).reset_index(drop=True)
df['log_amt'] = np.log1p(df['amount'])
gg = df.groupby(['region', 'industry_group'])
df['lag7'] = gg['log_amt'].shift(7)
df['lag14'] = gg['log_amt'].shift(14)
df['roll7'] = gg['log_amt'].transform(lambda s: s.shift(1).rolling(7).mean())

off_mean = df['log_amt'].where(df['is_offday'])
wk_mean = df['log_amt'].where(~df['is_offday'])
df['_off'] = off_mean; df['_wk'] = wk_mean
gg = df.groupby(['region', 'industry_group'])
df['roll7_offday_gap'] = (gg['_off'].transform(lambda s: s.shift(1).rolling(7, min_periods=1).mean())
                          - gg['_wk'].transform(lambda s: s.shift(1).rolling(7, min_periods=1).mean()))
df = df.drop(columns=['_off', '_wk'])

df['dow_num'] = df['date'].dt.dayofweek
df['week'] = df['date'].dt.isocalendar().week.astype(int)
hol = df.drop_duplicates('date').set_index('date')['is_holiday']
df['holiday_prev'] = df['date'].map((hol.shift(1, freq='D')).reindex(hol.index).fillna(False)).astype(int)
df['holiday_next'] = df['date'].map((hol.shift(-1, freq='D')).reindex(hol.index).fillna(False)).astype(int)
for c in ['is_weekend', 'is_holiday', 'is_offday', 'is_payday_week']:
    df[c] = df[c].astype(int)
df['use_for_model'] = df['lag14'].notna() & df['roll7'].notna()
print(f"학습 사용 행 {df['use_for_model'].sum():,} / 전체 {len(df):,} "
      f"(첫 사용일 {df.loc[df['use_for_model'], 'date'].min():%Y-%m-%d})")

학습 사용 행 5,100 / 전체 5,520 (첫 사용일 2025-07-15)


## 6. 지역 × 업종 구조 변수 (unit profile)

EDA에서 피해율이 업종의 **수요 구조**와 관련이 있었다 (춘천 안에서 외지 고객 비중이 높을수록 강한 비 피해율이 큼).
지역 × 업종(30개 단위)마다 변하지 않는 구조 변수를 만든다. 피해 예측의 상호작용 재료이자, 지원금 최적화의 **단위 정보**다.

| 변수 | 정의 | 출처 |
|---|---|---|
| `n_stores` | 점포 수 | 상가정보 |
| `chain_share`, `indep_share` | 체인 점포 비중, 독립점포 비중(1 − 체인) | 상가정보 |
| `outsider_share` | 외지 고객 매출 비중 | 카드 데이터2 |
| `elder_share_customer` | 60대 이상 고객 매출 비중 | 카드 데이터1 |
| `elder_share_flow` | 지역 60대 이상 유동인구 비중 | SKT |
| `evening_share` | 18~23시 매출 비중 | 카드 데이터1 |
| `offday_ratio` | 휴일 일평균 ÷ 평일 일평균 매출 | 카드 데이터1 |
| `sales_monthly` | 월평균 매출 (신한카드 기준) | 카드 데이터1 |
| `sales_per_store_monthly` | 점포당 월평균 매출 = sales_monthly ÷ n_stores | 카드 + 상가 |
| `base_daily_sales` | 기준일(비 없음·폭염 아님) 일평균 매출 | 카드 + 기상 |

> 구조 변수는 **6개월 전체**로 계산한 고정값이다. 피해 예측을 시간 순서로 검증할 때 미래 정보가 섞이는 문제는 이 값들이 '업종의 성격'을 나타내는 느리게 변하는 값이라 작다고 보고, 예측 단계에서 학습 기간만으로 다시 계산해 비교한다.

In [9]:
KEYS = ['region', 'industry_group']
o = origin[origin['customer_origin'] != '미상'].groupby(KEYS + ['customer_origin'], observed=True)['amount'].sum().unstack()
unit = pd.DataFrame({'outsider_share': o['외지'] / o.sum(axis=1)})

card1 = load('card1_clean')
p = card1[(card1['customer_type'] == '개인') & card1['analysis_include']]
el = p.assign(elder=p['age'] >= 60).groupby(KEYS + ['elder'])['amount'].sum().unstack()
unit['elder_share_customer'] = el[True] / el.sum(axis=1)
del card1, p

unit['evening_share'] = share.set_index(KEYS)['sales_share_18_23']
d_off = mart.groupby(KEYS + ['is_offday'])['amount'].mean().unstack()
unit['offday_ratio'] = d_off[True] / d_off[False]
months = mart.groupby(KEYS + ['month'])['amount'].sum().groupby(level=[0, 1]).mean()
unit['sales_monthly'] = months
base_days = df[(df['rain_sum'] == 0) & (df['heat_33'] == 0) & (df['snow'] == 0) & (df['cold'] == 0)]
unit['base_daily_sales'] = base_days.groupby(KEYS)['amount'].mean()
unit = unit.reset_index().merge(store[KEYS + ['n_stores', 'chain_share']], on=KEYS, how='left')
unit['indep_share'] = 1 - unit['chain_share']
unit['sales_per_store_monthly'] = unit['sales_monthly'] / unit['n_stores']
unit = unit.merge(flow_rm.groupby('region')['share_age_60'].mean().rename('elder_share_flow').reset_index(), on='region')
unit = unit.merge(share, on=KEYS)
assert len(unit) == 30 and unit.notna().all().all()

view = unit.set_index(KEYS)[['n_stores', 'indep_share', 'outsider_share', 'elder_share_customer', 'evening_share',
                             'offday_ratio', 'sales_per_store_monthly']].copy()
view['sales_per_store_monthly'] = view['sales_per_store_monthly'] / 1e4
view.rename(columns={'sales_per_store_monthly': '점포당 월매출(만원)'}).round(2)

n_stores  indep_share  outsider_share  elder_share_customer  evening_share  offday_ratio  점포당 월매출(만원)
region industry_group                                                                                                       
강남구    간편식                 1474        0.650           0.320                 0.090          0.330         0.980      449.070
       교육                  5164        0.960           0.270                 0.040          0.290         0.720    3,002.960
       교통·자동차               474        0.870           0.450                 0.240          0.220         0.710    7,345.840
       기타소매                3122        0.880           0.710                 0.070          0.300         0.970    9,007.380
       대형유통                 452        0.830           0.620                 0.170          0.310         1.120   97,378.250
       미용·개인서비스            3629        0.910           0.270                 0.150          0.210         1.200      874.080
       숙박                   537        0.980           0.360                 0.240          0.340         1.500    3,144.570
       식료품소매                863        0.870           0.270                 0.260          0.240         0.670    2,376.150
       여가·오락               1384        0.890           0.220                 0.140          0.390         0.650    2,022.540
       음식점                 7256        0.800           0.340                 0.140          0.470         0.950    3,299.060
       의료·보건               3825        0.990           0.480                 0.210          0.070         0.360   12,933.100
       의류·잡화               3499        0.860           0.420                 0.130          0.230         1.140    3,438.180
       주점·유흥               1227        0.920           0.320                 0.070          0.350         0.690      560.520
       카페·디저트              2671        0.710           0.570                 0.120          0.190         1.080    2,627.100
       편의점                  803        0.540           0.300                 0.090          0.340         0.720    4,704.890
춘천시    간편식                  836        0.710           0.140                 0.080          0.390         1.290      204.900
       교육                  1218        0.960           0.040                 0.040          0.340         0.270      397.970
       교통·자동차               514        0.950           0.120                 0.290          0.160         0.780    6,854.550
       기타소매                1362        0.900           0.120                 0.260          0.150         1.170    1,055.810
       대형유통                 407        0.900           0.100                 0.270          0.270         1.340   10,016.120
       미용·개인서비스            1537        0.940           0.090                 0.170          0.210         1.210      281.340
       숙박                   818        1.000           0.760                 0.140          0.300         1.440      176.670
       식료품소매                490        0.940           0.180                 0.280          0.200         1.180    2,960.350
       여가·오락                688        0.920           0.600                 0.190          0.280         1.360    1,982.840
       음식점                 3333        0.890           0.240                 0.180          0.480         1.380    1,820.490
       의료·보건                579        0.980           0.100                 0.280          0.060         0.400    5,712.660
       의류·잡화               1033        0.870           0.120                 0.250          0.200         1.190      782.320
       주점·유흥                610        0.930           0.140                 0.170          0.460         1.490      227.920
       카페·디저트              1128        0.850           0.170                 0.130          0.210         1.300      448.960
       편의점                  373        0.620           0.180                 0.100          0.410         1.030    3,700.340

In [10]:
corr = unit[['indep_share', 'outsider_share', 'elder_share_customer', 'evening_share', 'offday_ratio']].assign(
    log_sales_per_store=np.log(unit['sales_per_store_monthly'])).corr(method='spearman')
print('구조 변수 간 스피어만 상관 (30개 단위) — 0.8 이상이면 하나만 쓰는 것을 검토')
corr.round(2)

구조 변수 간 스피어만 상관 (30개 단위) — 0.8 이상이면 하나만 쓰는 것을 검토


,indep_share,outsider_share,elder_share_customer,evening_share,offday_ratio,log_sales_per_store
indep_share,1.000,-0.170,0.350,-0.300,0.010,-0.080
outsider_share,-0.170,1.000,-0.240,0.120,0.010,0.280
elder_share_customer,0.350,-0.240,1.000,-0.540,0.180,0.290
evening_share,-0.300,0.120,-0.540,1.000,0.140,-0.230
offday_ratio,0.010,0.010,0.180,0.140,1.000,-0.380
log_sales_per_store,-0.080,0.280,0.290,-0.230,-0.380,1.000


- 구조 변수끼리의 상관은 모두 |ρ| < 0.6이라 서로 다른 정보를 담고 있다 → 모두 유지한다.
- **점포당 월매출**은 업종·지역에 따라 최대 약 550배 차이가 나고(강남 대형유통 9.7억원 ~ 춘천 숙박 177만원, 신한카드 기준), 6단계 지원금 최적화에서 '피해액 ÷ 매출'을 계산할 때 분모로 쓴다.

## 7. 인코딩·상호작용

| 종류 | 변수 | 용도 |
|---|---|---|
| 이진 인코딩 | `is_chuncheon` (춘천 = 1) | 지역이 2개라 0/1 하나로 충분 |
| 원-핫 인코딩 | `ind_*` (업종대분류 15개 → 14개 열, 기준 = 음식점) | 선형·회귀 모델용 |
| 정수 코드 | `region_code`, `industry_code` | 트리 모델(LightGBM)의 범주형 입력 |
| 상호작용 | `rain_x_chuncheon`, `rain_x_outsider`, `rain_exposure`, `heat_x_chuncheon`, `rain_heavy_x_offday` | EDA에서 확인한 이질성 (지역·외지 고객·영업시간·휴일) |

- 업종이 15개뿐이라 **목표 인코딩(target encoding)은 쓰지 않는다** (각 범주에 368일 표본이 있어 원-핫으로 충분하고, 목표 인코딩은 누수 위험이 있다).
- 기상 × 업종 상호작용은 15개 × 기상 변수 수만큼 늘어나므로 테이블에 미리 만들지 않고, 모델에서 **업종별 계수(회귀)** 또는 **트리 분기(LightGBM)**로 처리한다.

In [11]:
df = df.merge(unit[KEYS + ['outsider_share', 'indep_share', 'evening_share', 'offday_ratio'] +
                    [f'sales_share_{s}' for s in SLOTS]], on=KEYS, how='left')
df['rain_exposure'] = sum(df[f'sales_share_{s}'] * df[f'rain_{s}'] for s in SLOTS)
df['log_rain_exposure'] = np.log1p(df['rain_exposure'])
df = df.drop(columns=[f'sales_share_{s}' for s in SLOTS])

df['is_chuncheon'] = (df['region'] == '춘천시').astype(int)
df['region_code'] = df['region'].map({'강남구': 0, '춘천시': 1})
groups = sorted(df['industry_group'].unique())
df['industry_code'] = df['industry_group'].map({g: i for i, g in enumerate(groups)})
dummies = pd.get_dummies(df['industry_group'], prefix='ind', dtype=int).drop(columns='ind_음식점')
df = pd.concat([df, dummies], axis=1)

df['rain_x_chuncheon'] = df['log_rain'] * df['is_chuncheon']
df['rain_x_outsider'] = df['log_rain'] * df['outsider_share']
df['heat_x_chuncheon'] = df['heat_33'] * df['is_chuncheon']
df['rain_heavy_x_offday'] = df['rain_heavy'] * df['is_offday']

ex = df[(df['date'] == '2025-09-16') & (df['region'] == '춘천시')][['industry_group', 'rain_sum'] + [f'rain_{s}' for s in SLOTS] + ['rain_exposure']]
print('예시: 한 날짜·지역의 일강수량과 업종별 영업시간 강수 노출')
ex.round(1)

예시: 한 날짜·지역의 일강수량과 업종별 영업시간 강수 노출


,industry_group,rain_sum,rain_00_05,rain_06_11,rain_12_17,rain_18_23,rain_exposure
2837,간편식,14.600,0.000,0.000,14.200,0.400,6.200
3021,교육,14.600,0.000,0.000,14.200,0.400,7.400
3205,교통·자동차,14.600,0.000,0.000,14.200,0.400,7.400
3389,기타소매,14.600,0.000,0.000,14.200,0.400,9.400
3573,대형유통,14.600,0.000,0.000,14.200,0.400,8.100
3757,미용·개인서비스,14.600,0.000,0.000,14.200,0.400,8.900
3941,숙박,14.600,0.000,0.000,14.200,0.400,6.500
4125,식료품소매,14.600,0.000,0.000,14.200,0.400,8.000
4309,여가·오락,14.600,0.000,0.000,14.200,0.400,7.800
4493,음식점,14.600,0.000,0.000,14.200,0.400,6.000


In [12]:
# 영업시간 강수 노출이 일강수량보다 매출 변화를 더 잘 설명하는가 (간단 점검: 주 고정효과 회귀의 설명력 비교)
import statsmodels.formula.api as smf
chk = df[df['use_for_model']].copy()
chk['week'] = chk['week'].astype(str)
rows = []
for r in ['강남구', '춘천시']:
    d = chk[chk['region'] == r]
    base = 'log_amt ~ C(industry_group) * (C(dow_num) + C(week)) + is_holiday + is_payday_week'
    for name, term in [('강수 없음', ''), ('일강수량 log', ' + C(industry_group):log_rain'),
                       ('영업시간 강수 노출 log', ' + C(industry_group):log_rain_exposure')]:
        res = smf.ols(base + term, data=d).fit()
        rows.append({'region': r, '강수 변수': name, 'adj_R2': res.rsquared_adj, 'AIC': res.aic})
pd.DataFrame(rows).pivot(index='강수 변수', columns='region', values=['adj_R2', 'AIC']).round(4)

adj_R2              AIC         
region            강남구   춘천시        강남구      춘천시
강수 변수                                          
강수 없음           0.984 0.972 -1,213.201 -280.612
영업시간 강수 노출 log  0.985 0.974 -1,252.430 -419.857
일강수량 log        0.984 0.973 -1,234.003 -366.404

- 같은 날 같은 지역이라도 업종마다 영업시간 강수 노출이 다르다. 예) 춘천 9월 16일에는 낮(12~17시)에 비가 왔는데, 낮에 장사하는 의류·잡화는 9.7mm, 밤에 장사하는 주점·유흥은 0.3mm로 계산된다.
- 업종별 강수 계수를 넣은 회귀에서 **영업시간 강수 노출이 일강수량보다 AIC가 낮다** (두 지역 모두, 특히 춘천 −420 vs −366). 피해 예측에서는 일강수량 대신 **영업시간 강수 노출**을 기본 강수 변수로 쓴다.

## 8. 최종 테이블 점검·저장

In [13]:
FEATURES = {
    # 식별·목표
    'date': ('식별', '날짜'), 'region': ('식별', '지역'), 'industry_group': ('식별', '업종대분류'),
    'amount': ('목표', '일 매출(원, 신한카드 개인)'), 'log_amt': ('목표', 'log(매출+1) — 모델 목표'),
    'use_for_model': ('식별', '학습 사용 여부 (lag14 존재)'),
    # 달력
    'dow_num': ('달력', '요일 0=월 ~ 6=일'), 'is_weekend': ('달력', '주말'), 'is_holiday': ('달력', '공휴일'),
    'is_offday': ('달력', '휴일(주말+공휴일)'), 'holiday_prev': ('달력', '전날이 공휴일'), 'holiday_next': ('달력', '다음날이 공휴일'),
    'is_payday_week': ('달력', '25일~말일'), 'week': ('달력', 'ISO 주 번호'), 'month': ('달력', '월'),
    # 매출 수준
    'lag7': ('매출 수준', '7일 전 log 매출'), 'lag14': ('매출 수준', '14일 전 log 매출'),
    'roll7': ('매출 수준', '직전 7일 log 매출 평균'), 'roll7_offday_gap': ('매출 수준', '직전 7일 휴일 − 평일 log 매출'),
    # 기상 원값(대표)
    'feels_like_max': ('기상', '일최고 체감온도(℃)'), 'temp_min': ('기상', '일최저기온(℃)'), 'wind_mean': ('기상', '평균 풍속(m/s)'),
    'rain_sum': ('기상', '일강수량(mm)'), 'snow_new': ('기상', '신적설(cm)'),
    # 기상 파생
    'heat_30_33': ('기상 파생', '체감 30~33℃'), 'heat_33': ('기상 파생', '체감 33℃ 이상'), 'heat_excess': ('기상 파생', '체감 30℃ 초과분'),
    'cold': ('기상 파생', '최저 −12℃ 이하'), 'cold_excess': ('기상 파생', '최저 −5℃ 미만분'),
    'rain_light': ('기상 파생', '강수 0~5mm'), 'rain_mid': ('기상 파생', '강수 5~30mm'), 'rain_heavy': ('기상 파생', '강수 30mm 이상'),
    'log_rain': ('기상 파생', 'log(1+일강수량)'), 'snow': ('기상 파생', '신적설 0.5cm 이상'), 'snow_cover': ('기상 파생', '적설이 남은 날'),
    'anom_feels_like': ('기상 파생', '체감온도 평년 편차'), 'anom_temp_min': ('기상 파생', '최저기온 평년 편차'), 'anom_rain': ('기상 파생', '강수량 평년 편차'),
    'rain_00_05': ('기상 파생', '00~05시 강수(mm)'), 'rain_06_11': ('기상 파생', '06~11시 강수'), 'rain_12_17': ('기상 파생', '12~17시 강수'), 'rain_18_23': ('기상 파생', '18~23시 강수'),
    'rain_exposure': ('기상 파생', '업종 시간대 매출비중 가중 강수(mm)'), 'log_rain_exposure': ('기상 파생', 'log(1+영업시간 강수 노출)'),
    'rain_prev': ('기상 파생', '전날 강수(mm)'), 'log_rain_prev': ('기상 파생', 'log(1+전날 강수)'),
    'rain_streak': ('기상 파생', '연속 강수일'), 'heat_streak': ('기상 파생', '연속 폭염일'),
    # 레이블
    'weather_type': ('레이블', '기상 유형'), 'heat_alert': ('레이블', '폭염 특보 수준 0/1/2'), 'rain_alert': ('레이블', '호우 특보 수준'),
    'cold_alert': ('레이블', '한파 특보 수준'), 'snow_alert': ('레이블', '대설 특보 수준'), 'alert_level': ('레이블', '특보 수준 최댓값'),
    # 구조
    'outsider_share': ('구조', '외지 고객 매출 비중'), 'indep_share': ('구조', '독립점포 비중'),
    'evening_share': ('구조', '18~23시 매출 비중'), 'offday_ratio': ('구조', '휴일/평일 매출'),
    # 인코딩·상호작용
    'is_chuncheon': ('인코딩', '춘천=1'), 'region_code': ('인코딩', '지역 정수 코드'), 'industry_code': ('인코딩', '업종 정수 코드'),
    'rain_x_chuncheon': ('상호작용', 'log_rain × 춘천'), 'rain_x_outsider': ('상호작용', 'log_rain × 외지 비중'),
    'heat_x_chuncheon': ('상호작용', 'heat_33 × 춘천'), 'rain_heavy_x_offday': ('상호작용', 'rain_heavy × 휴일'),
}
ind_cols = [c for c in df.columns if c.startswith('ind_')]
for c in ind_cols:
    FEATURES[c] = ('인코딩', f"업종 원-핫 ({c[4:]}, 기준=음식점)")
model = df[list(FEATURES)].copy()

checks = {
    '행 수 = 184일 × 2지역 × 15업종': len(model) == 184 * 2 * 15,
    '키 중복 없음': not model.duplicated(['date', 'region', 'industry_group']).any(),
    '매출 합계 보존': model['amount'].sum() == mart['amount'].sum(),
    '학습 사용 행에 결측 없음': model.loc[model['use_for_model']].drop(columns=['roll7_offday_gap']).notna().all().all(),
    '원-핫 합 ≤ 1': (model[ind_cols].sum(axis=1) <= 1).all(),
    'lag7은 7일 전 값과 일치 (누수 점검)': bool(np.allclose(
        model.sort_values(['region', 'industry_group', 'date']).groupby(['region', 'industry_group'])['log_amt'].shift(7).dropna(),
        model.sort_values(['region', 'industry_group', 'date'])['lag7'].dropna())),
}
for k, ok in checks.items():
    print(f"[{'OK' if ok else '실패'}] {k}")
assert all(checks.values())
missing = model.isna().sum()
print('\n결측이 있는 변수 (모두 기간 초반 lag/rolling 때문):', missing[missing > 0].to_dict())

[OK] 행 수 = 184일 × 2지역 × 15업종
[OK] 키 중복 없음
[OK] 매출 합계 보존
[OK] 학습 사용 행에 결측 없음
[OK] 원-핫 합 ≤ 1
[OK] lag7은 7일 전 값과 일치 (누수 점검)

결측이 있는 변수 (모두 기간 초반 lag/rolling 때문): {'lag7': 210, 'lag14': 420, 'roll7': 210, 'roll7_offday_gap': 180}


In [14]:
model.to_parquet(PROC / 'model_daily_group.parquet', index=False)
unit.to_parquet(PROC / 'unit_profile.parquet', index=False)
fd = pd.DataFrame([(k, v[0], v[1]) for k, v in FEATURES.items()], columns=['feature', 'group', 'description'])
fd.to_csv(REF / 'feature_dictionary.csv', index=False, encoding='utf-8-sig')
print(f'model_daily_group: {model.shape} | unit_profile: {unit.shape} | 변수 사전 {len(fd)}개')
fd.groupby('group').size().rename('변수 수').to_frame().T

model_daily_group: (5520, 79) | unit_profile: (30, 17) | 변수 사전 79개


group,구조,기상,기상 파생,달력,레이블,매출 수준,목표,상호작용,식별,인코딩
변수 수,4,5,24,9,6,4,2,4,4,17


## 정리

| 구분 | 결과 |
|---|---|
| 기상 변수 정리 | 원값 11개(VIF 최대 466) → 대표 5개 → 구간·평년 편차로 변환해 VIF < 10 |
| 새 기상 변수 | 강수 구간·log, **시간대별 강수**, **영업시간 강수 노출**(업종 매출 시간대로 가중), 전날 비, 연속 강수·폭염일, 평년 편차 |
| 레이블 | 기상 유형 6종, 폭염·호우·한파·대설 특보 수준(0/1/2), 종합 특보 수준 |
| 누수 방지 | 매출 수준 변수는 과거 값만 사용 (lag7·lag14·직전 7일 평균), 첫 14일은 학습 제외 표시 |
| 인코딩 | 지역 이진, 업종 원-핫(선형용)·정수 코드(트리용), 목표 인코딩은 쓰지 않음 |
| 상호작용 | 강수 × 춘천, 강수 × 외지 비중, 폭염 × 춘천, 강한 비 × 휴일 |
| 구조 변수 (30개 단위) | 점포 수, 독립점포·외지 고객·고령 고객·저녁 매출 비중, 휴일/평일 비, 점포당 월매출, 기준일 매출 |
| 산출물 | `model_daily_group.parquet` (5,520 × 79), `unit_profile.parquet` (30 × 17), `references/feature_dictionary.csv` (변수 79개) |
| 다음 단계 | **05 피해 예측** — 같은 모델에 '실제 날씨'와 '평소 날씨'를 넣어 업종별 피해율·피해액을 계산 |